In [1]:
from pathlib import Path

MODEL_ID = "google/gemma-4-E2B-it"
WEIGHTS  = Path("model_weights")

In [2]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Run our PyTorch-side tokenizer
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from architecture.tokenization import GemmaTokenizer

tok = GemmaTokenizer(WEIGHTS / "tokenizer.json")

s = "Explain MoE in transformers in 3 sentences."
ids = tok.encode(s)

print(f"vocab_size = {tok.vocab_size:,}")
print(f"ids ({len(ids)}): {ids[:10]} ...")

for tid, piece in tok.pretty(ids)[:10]:
    print(f"  {tid:>6}  {piece!r}")

print(f"\nroundtrip: {tok.decode(ids)!r}")

vocab_size = 262,144
ids (10): [155122, 7945, 236788, 528, 39687, 528, 236743, 236800, 23974, 236761] ...
  155122  'Explain'
    7945  '▁Mo'
  236788  'E'
     528  '▁in'
   39687  '▁transformers'
     528  '▁in'
  236743  '▁'
  236800  '3'
   23974  '▁sentences'
  236761  '.'

roundtrip: 'Explain MoE in transformers in 3 sentences.'


In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Embed via OUR GemmaEmbedding  (pure nn.Embedding lookup)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.embedding import GemmaEmbedding

emb  = GemmaEmbedding.from_safetensors(WEIGHTS / "model.safetensors")
ours = emb(torch.tensor(ids, dtype=torch.long))

print(f"weight : {tuple(emb.embed.weight.shape)}  dtype={emb.embed.weight.dtype}")
print(f"vecs   : {tuple(ours.shape)}")
print(f"ours[0, :6] = {ours[0, :6].tolist()}")

weight : (262144, 1536)  dtype=torch.bfloat16
vecs   : (10, 1536)
ours[0, :6] = [-0.1123046875, 0.1826171875, -0.08251953125, -0.7265625, 2.171875, -1.484375]


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Image + audio soft-token embeddings (towers borrowed from HF for now)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from architecture.embedding import GemmaVisionProjector, GemmaAudioProjector

shard = WEIGHTS / "model.safetensors"
vproj = GemmaVisionProjector.from_safetensors(shard)
aproj = GemmaAudioProjector.from_safetensors(shard)

img_embeds = vproj.embed_file("test_data/Moon.jpg",  MODEL_ID)
aud_embeds = aproj.embed_file("test_data/Chats.mp3", MODEL_ID)

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Inspect image + audio soft-token embeddings (batch-dim agnostic)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print(f"Moon.jpg  → {tuple(img_embeds.shape)}  dtype={img_embeds.dtype}")
print(f"  first token : {img_embeds.flatten(0, -2)[0, :6].tolist()}")

print(f"Chats.mp3 → {tuple(aud_embeds.shape)}  dtype={aud_embeds.dtype}")
print(f"  first token : {aud_embeds.flatten(0, -2)[0, :6].tolist()}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/models/gemma4/feature_extraction_gemma4.py:208: RuntimeWarning: divide by zero encountered in matmul
  mel_spec = np.matmul(magnitude_spec, self.mel_filters)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/transformers/models/g

Moon.jpg  → (260, 1536)  dtype=torch.bfloat16
  first token : [0.9453125, 0.1572265625, -0.88671875, -1.5546875, -0.859375, 0.314453125]
Chats.mp3 → (1, 135, 1536)  dtype=torch.bfloat16
  first token : [-1.90625, 0.037841796875, -3.8125, -1.3203125, -1.7578125, 0.8046875]


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  RMSNorm — load the final-norm weight from the shard, run a tensor through.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import torch
from architecture.RMSnorm import GemmaRMSNorm

norm = GemmaRMSNorm.from_safetensors(
    WEIGHTS / "model.safetensors",
    "model.language_model.norm.weight",
)

torch.manual_seed(0)
x = torch.randn(1, 8, 1536, dtype=torch.bfloat16)
y = norm(x)

print(f"weight  : {tuple(norm.weight.shape)}  dtype={norm.weight.dtype}")
print(f"x       : {tuple(x.shape)}  dtype={x.dtype}")
print(f"y       : {tuple(y.shape)}  dtype={y.dtype}")
print(f"y[0,0,:6] = {y[0, 0, :6].tolist()}")

weight  : (1536,)  dtype=torch.bfloat16
x       : (1, 8, 1536)  dtype=torch.bfloat16
y       : (1, 8, 1536)  dtype=torch.bfloat16
y[0,0,:6] = [20.125, 2.59375, 14.125, 26.125, 7.59375, -20.5]


In [6]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  RoPE — emit (cos, sin) tables for both layer types.
#  No weights to load; everything is determined by config numbers.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from architecture.rope import rope_local, rope_global

seq_len = 8
position_ids = torch.arange(seq_len, dtype=torch.long)[None]   # (1, S)

# global: head_dim=512, theta=1e6, only first 25% of dims rotate
x512 = torch.zeros(1, seq_len, 512, dtype=torch.bfloat16)
cos_g, sin_g = rope_global()(x512, position_ids)

# local: head_dim=256, theta=10_000, all dims rotate
x256 = torch.zeros(1, seq_len, 256, dtype=torch.bfloat16)
cos_l, sin_l = rope_local()(x256, position_ids)

print(f"global cos: {tuple(cos_g.shape)}  cos[0,1,:6] = {cos_g[0, 1, :6].tolist()}")
print(f"global sin: {tuple(sin_g.shape)}  sin[0,1,:6] = {sin_g[0, 1, :6].tolist()}")
print(f"  → tail (no-rope dims) cos[0,1,-4:] = {cos_g[0, 1, -4:].tolist()}  (should be ~1)")
print(f"  → tail (no-rope dims) sin[0,1,-4:] = {sin_g[0, 1, -4:].tolist()}  (should be ~0)")
print()
print(f"local  cos: {tuple(cos_l.shape)}  cos[0,1,:6] = {cos_l[0, 1, :6].tolist()}")
print(f"local  sin: {tuple(sin_l.shape)}  sin[0,1,:6] = {sin_l[0, 1, :6].tolist()}")

global cos: (1, 8, 512)  cos[0,1,:6] = [0.5390625, 0.58203125, 0.625, 0.66015625, 0.69140625, 0.72265625]
global sin: (1, 8, 512)  sin[0,1,:6] = [0.83984375, 0.8125, 0.78125, 0.75, 0.72265625, 0.69140625]
  → tail (no-rope dims) cos[0,1,-4:] = [1.0, 1.0, 1.0, 1.0]  (should be ~1)
  → tail (no-rope dims) sin[0,1,-4:] = [0.0, 0.0, 0.0, 0.0]  (should be ~0)

local  cos: (1, 8, 256)  cos[0,1,:6] = [0.5390625, 0.59765625, 0.6484375, 0.69140625, 0.73046875, 0.765625]
local  sin: (1, 8, 256)  sin[0,1,:6] = [0.83984375, 0.80078125, 0.76171875, 0.72265625, 0.6796875, 0.640625]
